# The decoherence dial — how much you lose is how much it learned

**The punchline.** Decoherence is not a switch and it is not damage. Couple a qubit to
one other qubit by an angle $\theta$ and the superposition fades *continuously*, and the
amount that fades is exactly the amount by which the other qubit's two possible states
have become distinguishable. Visibility $V$ and which-path distinguishability $D$ obey

$$V^2 + D^2 = 1$$

at every setting of the dial. Nothing pushed the qubit. Nothing random happened. The
environment learned something, and the superposition died of the learning.

Background: **[06 — Why the world looks classical](../06-decoherence.ipynb)** §3–§7 builds
this machinery. This exhibit turns the knob and plots what comes out.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsim import Circuit, viz
from qsim.decoherence import dephasing_coupling
from qsim.gates import H

np.set_printoptions(precision=3, suppress=True)

## 1. The coupling, in one sentence

`dephasing_coupling(q, env, theta=t)` is a single controlled rotation: the environment
qubit starts in $\lvert 0\rangle$ and is rotated by $t$ **only where `q` is
$\lvert 1\rangle$**. So

$$a\lvert 0\rangle + b\lvert 1\rangle \;\longrightarrow\;
a\lvert 0\rangle\lvert E_0\rangle + b\lvert 1\rangle\lvert E_1\rangle,
\qquad \langle E_0 \vert E_1\rangle = \cos(t/2).$$

Read $\lvert E_0\rangle$ and $\lvert E_1\rangle$ as the environment's two answers to the
question "was the qubit 1?". At $t = 0$ the two answers are the same state: the
environment says the same thing either way, so it has learned nothing. At $t = \pi$ they
are orthogonal: a perfect record. In between, a partial and unreliable one.

Marking the extra qubit with `qc.environment(1)` **traces nothing out**. The qubit stays
in the state tensor, the global state stays exactly pure forever, and nothing stochastic
happens anywhere. The marking only tells `qc.inspect` which qubits to stop tracking when
you ask for the *system's* point of view. Decoherence lives in that choice of view.

In [ ]:
def dephased(theta: float) -> Circuit:
    """|+> on one qubit, then let one environment qubit look at it by angle theta."""
    qc = Circuit(name="dial", seed=2024)
    q = qc.alloc("q")
    env = qc.environment(1)
    H(q)                                    # a maximal superposition: coherence 0.5
    dephasing_coupling(q, env[0], theta=theta)
    return qc


demo = dephased(np.pi / 3)
q_demo = demo.qubits[0]

print("the whole two-qubit state (still perfectly pure):")
print("   ", demo.inspect.ket())
print("    entropy of the whole system:",
      round(demo.inspect.entanglement_entropy(list(demo.qubits)), 12), "bits")
print()
print("the system's point of view (the environment qubit ignored):")
print(demo.inspect.system_density_matrix())
print(f"    coherence |rho_01|  = {demo.inspect.coherence(q_demo):.6f}   (0.5 for a fresh |+>)")
print(f"    Bloch vector        = {np.round(demo.inspect.bloch_vector(q_demo), 6)}")
print(f"    system entropy      = {demo.inspect.system_entropy():.6f} bits")

Zero entropy for the pair, non-zero entropy for the system. Those are the same state
described twice, once with the environment qubit in the account and once without.

Note also which numbers moved. The diagonal of $\rho$ — the two probabilities — is
untouched at $(0.5, 0.5)$. Only the off-diagonal shrank. The environment did not push the
qubit toward one answer or the other; it made the two answers stop interfering.

## 2. Turning the knob

Now sweep $\theta$ from 0 to $\pi$ and measure three things at each setting.

**Coherence** $|\rho_{01}|$ and the **Bloch $x$ component** read the system's density
matrix directly — a simulator cheat, but the cleanest view.

**Visibility** is the honest experimental version: run the interferometer of
[one_qubit_playground](one_qubit_playground.ipynb) — $H$, couple, $H$ — and take
$V = P(0) - P(1)$ at the end. That is a number you could measure in a laboratory by
counting clicks.

**Distinguishability** is the environment's side of the ledger: how well could someone
holding the environment qubit tell $\lvert E_0\rangle$ from $\lvert E_1\rangle$?

In [ ]:
def which_path_distinguishability(qc: Circuit) -> float:
    """How distinguishable the environment's two conditional states are, from 0 to 1.

    The state tensor has shape (2, 2): axis 0 is the system qubit, axis 1 the
    environment. Fixing axis 0 to 0 or 1 slices out the (unnormalised) environment
    state that goes with each branch of the system.
    """
    psi = qc.inspect.state_tensor()
    e0, e1 = psi[0], psi[1]
    # np.vdot conjugates its first argument, so this is <e0|e1>; dividing by the two
    # norms turns the unnormalised slices into a proper overlap between unit vectors.
    overlap = np.vdot(e0, e1) / (np.linalg.norm(e0) * np.linalg.norm(e1))
    # Two states with overlap magnitude c can be told apart with certainty exactly to
    # the extent that c falls short of 1; sqrt(1 - c^2) is the standard measure.
    return float(np.sqrt(max(0.0, 1.0 - abs(overlap) ** 2)))


thetas = np.linspace(0.0, np.pi, 121)
coherence = []
bloch_x = []
populations = []
visibility = []
distinguishability = []
system_entropy = []

for theta in thetas:
    qc = dephased(theta)
    q = qc.qubits[0]
    coherence.append(qc.inspect.coherence(q))
    bloch_x.append(qc.inspect.bloch_vector(q)[0])
    rho = qc.inspect.system_density_matrix()
    populations.append(float(np.real(rho[0, 0])))
    distinguishability.append(which_path_distinguishability(qc))
    system_entropy.append(qc.inspect.system_entropy())

    # The same experiment with the interferometer closed: H, couple, H, then read
    # the two outcome probabilities. This is what a laboratory would actually see.
    closed = dephased(theta)
    H(closed.qubits[0])
    probs = np.real(np.diag(closed.inspect.system_density_matrix()))
    visibility.append(float(probs[0] - probs[1]))

Six lists, 121 entries each, and no plot yet — because it is worth saying in advance what
the two panels are for.

The **left** panel puts the three coherence readouts on the same axes as the prediction
$\cos(\theta/2)$, plus one control: the probability of finding the qubit in
$\lvert 0\rangle$ *before* the interferometer is closed. If decoherence were the
environment jostling the qubit, that control would move. Watch whether it does.

The **right** panel is the ledger: visibility against distinguishability, and their
squares added together.

In [ ]:
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(11.2, 3.8))

ax_left.plot(thetas, visibility, lw=2.4, color="crimson",
             label="visibility  P(0) − P(1)")
ax_left.plot(thetas, bloch_x, lw=1.6, color="teal", label="Bloch x of the system")
ax_left.plot(thetas, np.array(coherence) * 2.0, lw=1.6, color="darkorange",
             label=r"$2\,|\rho_{01}|$")
ax_left.plot(thetas, np.cos(thetas / 2), "k--", lw=1.2, label=r"predicted $\cos(\theta/2)$")
ax_left.plot(thetas, populations, lw=1.4, color="gray", label="P(0) before recombining")
ax_left.set_ylim(-0.05, 1.35)
ax_left.set_title("three readouts, one curve")

ax_right.plot(thetas, visibility, lw=2.4, color="crimson", label="visibility $V$")
ax_right.plot(thetas, distinguishability, lw=2.4, color="teal",
              label="distinguishability $D$")
ax_right.plot(thetas, np.array(visibility) ** 2 + np.array(distinguishability) ** 2,
              "k--", lw=1.4, label="$V^2 + D^2$")
ax_right.plot(thetas, system_entropy, lw=1.4, color="purple",
              label="system entropy (bits)")
ax_right.set_ylim(-0.05, 1.35)
ax_right.set_title("a conservation law: what you lose, it gained")

for ax in (ax_left, ax_right):
    ax.set_xlabel(r"$\theta$ — how hard the environment looks")
    ax.set_xticks([0, np.pi / 2, np.pi], ["0", "π/2", "π"])
    ax.legend(fontsize=8, loc="lower left")
fig.tight_layout()

**Left.** Four different questions, one answer: $\cos(\theta/2)$. The laboratory
visibility, the Bloch $x$ component, and twice the density matrix's off-diagonal all lie
on the predicted curve. And the flat grey line is the part that is easy to miss — the
probability of finding the qubit in $\lvert 0\rangle$ *before* recombining never budges
from $0.5$. Whatever the environment did, it did not push the qubit around.

**Right.** The conservation law. As visibility falls, distinguishability rises, and
$V^2 + D^2 = 1$ to floating-point precision at every angle. This is **complementarity**
as an accounting identity rather than a slogan: interference and which-path information
are two ways of spending one budget. You do not have to *read* the environment for the
interference to go; it is enough that the information is *there* to be read.

The purple curve is the same story in bits. The system's entropy — zero when it has a
state of its own, one bit when it is maximally entangled with the environment — rises
exactly as the record becomes reliable. And because the global state stays pure, that
number is *also* the environment's entropy: the information is not gone, it is in the
correlation.

## 3. The knob, drawn

`viz.dephasing_panels` puts three views of one setting side by side: the Bloch vector
retracting toward the origin, the marker sliding down the visibility curve, and the
density matrix with its diagonal fixed and its off-diagonal fading.

In [ ]:
fig_panels = viz.dephasing_panels(2.0 * np.pi / 3.0)

(`06-decoherence.ipynb` has this as a live slider, `viz.interact_dephasing()`. Sliders and
a kernel with no browser attached deadlock each other, so this exhibit takes a still.)

## Where to go next

- **[quantum_eraser](quantum_eraser.ipynb)**: turn the dial all the way to $\pi$, then
  turn it back, and watch the interference return exactly. Nothing was lost, because
  nothing was thrown away.
- **[einselection](einselection.ipynb)**: the dial has a second knob — *which* basis the
  environment asks about — and that one decides which states look classical.
- **[wigners_friend](wigners_friend.ipynb)**: the same coupling, with the environment
  qubit renamed "an observer".

## Assertions

The claims above, re-checked numerically.

In [ ]:
# 1. The global state stays pure at every setting; only the *view* is mixed.
for theta in (0.0, 0.7, np.pi / 2, np.pi):
    qc = dephased(theta)
    assert np.isclose(qc.inspect.entanglement_entropy(list(qc.qubits)), 0.0, atol=1e-12)
    assert np.isclose(qc.inspect.norm(), 1.0)

# 2. Every readout is cos(theta/2).
assert np.allclose(visibility, np.cos(thetas / 2), atol=1e-12)
assert np.allclose(bloch_x, np.cos(thetas / 2), atol=1e-12)
assert np.allclose(np.array(coherence) * 2.0, np.cos(thetas / 2), atol=1e-12)

# 3. Populations never move: this is information, not disturbance.
assert np.allclose(populations, 0.5, atol=1e-12)

# 4. Complementarity, as an identity rather than an inequality.
assert np.allclose(np.array(visibility) ** 2 + np.array(distinguishability) ** 2,
                   1.0, atol=1e-12)

# 5. The ends of the dial: untouched at theta = 0, fully classical at theta = pi.
assert np.isclose(visibility[0], 1.0) and np.isclose(system_entropy[0], 0.0, atol=1e-12)
assert np.isclose(visibility[-1], 0.0, atol=1e-12) and np.isclose(system_entropy[-1], 1.0)

# 6. The system's entropy equals the environment's, because the whole is pure.
mid = dephased(1.1)
assert np.isclose(mid.inspect.system_entropy(), mid.inspect.environment_entropy())

print("all assertions passed")